# PIMRE: RbTiBi 全流程 Pipeline

**原始实验数据**: `/home/dengxw/ARPES/mapping/data/theory/dataCliu.h5` → `RbTiBi/map`

流程：
1. DFT 数据处理
2. 实验数据交互式校准（角度空间 + 动量空间）
3. MRF 能带重建
4. 结果可视化

> 校准步骤需要交互：拖拽红线/绿线到 Γ 点，关闭窗口继续。

## 环境与导入

In [ ]:
%matplotlib widget
import os, sys, json, math
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scipy.io as sio
from decimal import Decimal
from scipy.signal import savgol_filter

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from pimre.dft.reader import read_band_gap, read_dft_csv, expand_bz, interpolate_to_grid, save_band_map_mat
from pimre.experiment.calibration import Angle2Mon, KDInterp, RotateCoordinates, save_preprocessed_h5
from pimre.utils.io import loadHDF
from pimre.mrf.model import MrfRec
from pimre.mrf.symmetry import sym_band
from pimre.mrf.evaluation import expand_dft_bands, theory_data_expand
from pimre.kpath.symmetry import Get_G_M_K, dft_KM
from pimre.kpath.path import points2path, bandpath_map as bpm
from pimre.utils.interaction import DraggableVLine, DraggableHLine

print("Imports OK")

## 配置参数

In [ ]:
# 路径配置
DATA_BASE = "/home/dengxw/ARPES"
DFT_DIR  = os.path.join(DATA_BASE, "share_data", "RbTiBi", "Raw_Data")
EXP_H5   = os.path.join(DATA_BASE, "mapping", "data", "theory", "dataCliu.h5")
EXP_DSET = "RbTiBi/map"  # 嵌套 HDF5 路径
TEST_DIR = os.path.join("..", "test")
os.makedirs(TEST_DIR, exist_ok=True)

# 物理参数
CRYSTAL = [5.8077, 5.8077, 9.1297, 90, 90, 120]
WORK_FUNCTION = 16.03
NKX, NKY = 20, 20

# MRF 超参数
HYPERPARAMS = [
    {"index": 0, "k_scale": 1.24, "offset": 0.75, "eta": 0.0000065},
    {"index": 1, "k_scale": 1.24, "offset": 0.8,  "eta": 0.0000000065},
    {"index": 2, "k_scale": 1.0,  "offset": 0.76, "eta": 0.05},
    {"index": 3, "k_scale": 1.24, "offset": 0.63, "eta": 0.000000065},
    {"index": 4, "k_scale": 1.24, "offset": 0.62, "eta": 0.0045},
]

# 加载上次校准值
CALIB_FILE = os.path.join(TEST_DIR, "calibration.json")
if os.path.exists(CALIB_FILE):
    with open(CALIB_FILE) as f:
        calib = json.load(f)
    print(f"加载已有校准: kx_shift={calib['kx_shift']:.4f}, ky_shift={calib['ky_shift']:.4f}")
else:
    calib = {"kx_shift": 0.0, "ky_shift": 0.0, "ky_angle_offset": 0.0,
             "kx_grid_shift": 0.0, "ky_grid_shift": 0.0}
    print("无已有校准，使用默认值 0.0")

print("Config OK")

---
## Step 1: DFT 数据处理

In [ ]:
print("=" * 60)
print("Step 1: DFT Processing")
print("=" * 60)

fermi, vbm, cbm = read_band_gap(os.path.join(DFT_DIR, "BAND_GAP"))
print(f"Fermi = {fermi:.4f} eV, VBM = {vbm}, CBM = {cbm}")

cartesian_coords, energy_bands, ebands = read_dft_csv(
    os.path.join(DFT_DIR, "extracted_data.csv"), fermi, nkx=NKX, nky=NKY)
print(f"ebands shape: {ebands.shape}")

bz_coords, repeated_bands = expand_bz(cartesian_coords, energy_bands)
print(f"BZ points: {bz_coords.shape[0]}")

mapping, kx_grid, ky_grid = interpolate_to_grid(bz_coords, repeated_bands, nx=101, ny=101)
gap_id = vbm + 1
evb = mapping[:gap_id][::-1]
ecb = mapping[gap_id:]
print(f"evb: {evb.shape}, ecb: {ecb.shape}")

band_map_path = os.path.join(TEST_DIR, "band_map.mat")
save_band_map_mat(band_map_path, evb, ecb, kx_grid, ky_grid)
print(f"Saved: {band_map_path}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(evb[0], extent=(kx_grid.min(), kx_grid.max(), ky_grid.min(), ky_grid.max()),
               origin="lower", cmap="plasma")
ax.set(title="Top Valence Band (DFT)", xlabel=r"$k_x$ ($\AA^{-1}$)", ylabel=r"$k_y$ ($\AA^{-1}$)")
plt.colorbar(im, label="E-EF (eV)")
fig.savefig(os.path.join(TEST_DIR, "dft_top_valence.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Step 2: 实验数据加载

In [ ]:
print("=" * 60)
print("Step 2: Load Experimental Data")
print("=" * 60)

with h5py.File(EXP_H5, "r") as f:
    bands = f[EXP_DSET][:]
print(f"bands shape: {bands.shape} (E, kx_angle, ky_angle)")

# 构建坐标轴 — 参考原 notebook 中 RbTiBi 的参数
E_range = (1.06, 1.06 + (-0.01) * (bands.shape[0] - 1))
kx_range = (-20.3729, -20.3729 + 0.0448678 * (bands.shape[1] - 1))
ky_range = (-7.3251, -7.3251 + 1 * (bands.shape[2] - 1))

E_grid = np.linspace(*E_range, bands.shape[0])
kx_angle = np.linspace(*kx_range, bands.shape[1])
ky_angle = np.linspace(*ky_range, bands.shape[2])

print(f"E_grid:  [{E_grid[0]:.4f}, {E_grid[-1]:.4f}] ({len(E_grid)} pts)")
print(f"kx_angle: [{kx_angle[0]:.4f}, {kx_angle[-1]:.4f}] ({len(kx_angle)} pts)")
print(f"ky_angle: [{ky_angle[0]:.4f}, {ky_angle[-1]:.4f}] ({len(ky_angle)} pts)")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(bands[10], aspect="auto",
               extent=[ky_angle[0], ky_angle[-1], kx_angle[0], kx_angle[-1]],
               origin="lower", cmap="plasma")
ax.set(title="Raw Data (layer 10)", xlabel="ky angle (deg)", ylabel="kx angle (deg)")
plt.colorbar(im, label="Intensity")
plt.show()

---
## Step 3: 交互式 Gamma 点校准（角度空间）

**拖拽红色竖线和绿色横线到 Γ 点中心。** 关闭窗口后偏移值会显示在下方。

In [ ]:
CALIBRATION_LAYER = 10

print(f"当前校准值: kx_shift={calib['kx_shift']:.4f}, ky_shift={calib['ky_shift']:.4f}")
print("拖拽红/绿线到 Γ 点，然后关闭窗口...")

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlabel("ky angle (deg)")
ax.set_ylabel("kx angle (deg)")

im = ax.imshow(bands[CALIBRATION_LAYER], aspect="auto",
               extent=[ky_angle[0], ky_angle[-1], kx_angle[0], kx_angle[-1]],
               cmap="plasma", origin="lower")
plt.colorbar(im, ax=ax, label="Intensity")

cx = (ky_angle[0] + ky_angle[-1]) / 2
cy = (kx_angle[0] + kx_angle[-1]) / 2
vline = DraggableVLine(ax, x=cx, color="red", linestyle="--", linewidth=2)
hline = DraggableHLine(ax, y=cy, color="green", linestyle="--", linewidth=2)

ax.set_title(f"Layer {CALIBRATION_LAYER} — Drag lines to Γ point, then close")
plt.tight_layout(rect=[0, 0, 0.95, 0.95])
plt.grid(True)
plt.show()

In [ ]:
calib["kx_shift"] = hline.y
calib["ky_shift"] = vline.x
print(f"\nkx_shift = {calib['kx_shift']:.4f}   (horizontal line)")
print(f"ky_shift = {calib['ky_shift']:.4f}   (vertical line)")

with open(CALIB_FILE, "w") as f:
    json.dump(calib, f, indent=2)
print(f"Saved to {CALIB_FILE}")

---
## Step 4: 角度→动量转换 + 旋转扩展

In [ ]:
print("=" * 60)
print("Step 4: Angle-to-Momentum + Rotation")
print("=" * 60)

ky_angle = ky_angle - calib["ky_angle_offset"]
kx_angle = kx_angle - calib["kx_shift"]
ky_angle = ky_angle - calib["ky_shift"]
print(f"Applied: kx_shift={calib['kx_shift']:.4f}, ky_shift={calib['ky_shift']:.4f}")

KX, KY = Angle2Mon(E_grid, kx_angle, ky_angle, work_function=WORK_FUNCTION)
print(f"KX shape: {KX.shape}")

n_rot = 6
bands_rep = np.repeat(bands[:, :, np.newaxis], n_rot, axis=2)
bands_rep = bands_rep.reshape(bands.shape[0], bands.shape[1], -1)

KX_rot = np.zeros((KX.shape[0], KX.shape[1], KX.shape[2] * n_rot))
KY_rot = np.zeros_like(KX_rot)
KX_rot[:, :, :KX.shape[2]] = KX
KY_rot[:, :, :KY.shape[2]] = KY
for i in range(1, n_rot):
    kxr, kyr = RotateCoordinates(KX, KY, theta=60 * i)
    KX_rot[:, :, i * KX.shape[2]:(i + 1) * KX.shape[2]] = kxr
    KY_rot[:, :, i * KY.shape[2]:(i + 1) * KY.shape[2]] = kyr

n_out = np.max(KX_rot.shape)
kx_out = np.linspace(np.min(KX_rot), np.max(KX_rot), n_out)
ky_out = np.linspace(np.min(KY_rot), np.max(KY_rot), n_out)
kxm, kym = np.meshgrid(kx_out, ky_out, indexing="ij")
print(f"Output grid: {n_out}x{n_out}")

E_temp = KDInterp(bands_rep[CALIBRATION_LAYER],
                  KX_rot[CALIBRATION_LAYER], KY_rot[CALIBRATION_LAYER],
                  radius=0.05, kx_grid=kxm, ky_grid=kym)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(E_temp, extent=(ky_out[0], ky_out[-1], kx_out[0], kx_out[-1]),
               origin="lower", cmap="plasma")
ax.set(title="Momentum Space Preview", xlabel=r"$k_y$ ($\AA^{-1}$)", ylabel=r"$k_x$ ($\AA^{-1}$)")
ax.axvline(0, color="red", ls="--", lw=1)
ax.axhline(0, color="red", ls="--", lw=1)
plt.colorbar(im, label="Intensity")
plt.show()

---
## Step 5: 交互式网格偏移校准（动量空间）

**如果 Γ 点不在原点，拖拽线条到正确位置。** 关闭窗口后继续。

In [ ]:
print("拖拽红/绿线到 Γ 点（动量空间），然后关闭...")

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlabel(r"$k_y$ ($\AA^{-1}$)")
ax.set_ylabel(r"$k_x$ ($\AA^{-1}$)")
im = ax.imshow(E_temp, aspect="auto",
               extent=[ky_out[0], ky_out[-1], kx_out[0], kx_out[-1]],
               cmap="plasma", origin="lower")
plt.colorbar(im, ax=ax, label="Intensity")

cx = (ky_out[0] + ky_out[-1]) / 2
cy = (kx_out[0] + kx_out[-1]) / 2
vg = DraggableVLine(ax, x=cx, color="red", linestyle="--", linewidth=2)
hg = DraggableHLine(ax, y=cy, color="green", linestyle="--", linewidth=2)

ax.set_title("Drag lines to Γ point in momentum space, then close")
plt.tight_layout(rect=[0, 0, 0.95, 0.95])
plt.grid(True)
plt.show()

In [ ]:
calib["kx_grid_shift"] = hg.y
calib["ky_grid_shift"] = vg.x
print(f"\nkx_grid_shift = {calib['kx_grid_shift']:.4f}")
print(f"ky_grid_shift = {calib['ky_grid_shift']:.4f}")

with open(CALIB_FILE, "w") as f:
    json.dump(calib, f, indent=2)
print(f"Saved to {CALIB_FILE}")

---
## Step 6: KD-Interpolation（全层）

In [ ]:
print("=" * 60)
print("Step 6: KD-Interpolation")
print("=" * 60)

kx_out = kx_out - calib["kx_grid_shift"]
ky_out = ky_out - calib["ky_grid_shift"]
kxm, kym = np.meshgrid(kx_out, ky_out, indexing="ij")

print(f"KD-interp on {bands_rep.shape[0]} layers...")
E_Mon = np.zeros((bands_rep.shape[0], n_out, n_out))
for i in range(bands_rep.shape[0]):
    if i % 20 == 0:
        print(f"  layer {i}/{bands_rep.shape[0]}")
    E_Mon[i] = KDInterp(bands_rep[i], KX_rot[i], KY_rot[i],
                         radius=0.05, kx_grid=kxm, ky_grid=kym)

print(f"E_Mon shape: {E_Mon.shape}")

prep_path = os.path.join(TEST_DIR, "exp_preprocessed.h5")
save_preprocessed_h5(prep_path, E_grid, kx_out, ky_out, E_Mon)
print(f"Saved: {prep_path}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(E_Mon[10], extent=(ky_out[0], ky_out[-1], kx_out[0], kx_out[-1]),
               origin="lower", cmap="plasma")
ax.set(title="Preprocessed (layer 10)", xlabel=r"$k_y$ ($\AA^{-1}$)", ylabel=r"$k_x$ ($\AA^{-1}$)")
ax.axvline(0, color="red", ls="--", lw=1)
ax.axhline(0, color="red", ls="--", lw=1)
plt.colorbar(im, label="Intensity")
fig.savefig(os.path.join(TEST_DIR, "exp_layer10.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Step 7: MRF 能带重建

In [ ]:
print("=" * 60)
print("Step 7: MRF Reconstruction")
print("=" * 60)

data = loadHDF(prep_path)
E = data["E"]
kx = data["kx"][1:-1]
ky = data["ky"][1:-1]
I = np.transpose(data["V"][1:-1, 1:-1, :], (1, 2, 0))
print(f"MRF: E={E.shape}, kx={kx.shape}, ky={ky.shape}, I={I.shape}")

mrf = MrfRec(E=E, kx=kx, ky=ky, I=I, eta=0.12)
mrf.smoothenI(sigma=(0.5, 0.5, 0.5 / 5))

mat_data = sio.loadmat(band_map_path)
evb_m = mat_data["evb"][:]
ecb_m = mat_data["ecb"][:]
E_dft = np.nan_to_num(np.vstack((ecb_m[::-1], evb_m)))
E_dft = E_dft[33:]

if np.abs(np.sum(np.diff(mat_data["kxxsc"][:, 0]))) > np.abs(np.sum(np.diff(mat_data["kxxsc"][0, :]))):
    ky_dft = mat_data["kxxsc"][:, 0]
    kx_dft = mat_data["kyysc"][0, :]
else:
    ky_dft = mat_data["kxxsc"][0, :]
    kx_dft = mat_data["kyysc"][:, 0]
print(f"DFT bands: {E_dft.shape}")

E_dft_exp, kx_dft_exp, ky_dft_exp = expand_dft_bands(E_dft, kx_dft, ky_dft, kx, ky)
kx_dft = kx_dft_exp; ky_dft = ky_dft_exp; E_dft = E_dft_exp

G, M, M1, K, K1, K2, K3, K4, K5 = Get_G_M_K(CRYSTAL, kx, ky)
KP_dft, MP_dft = dft_KM(kx_dft, ky_dft)
print(f"G={G}, M={M}, K={K}")

kx_dft_0 = kx[M[0]] - ((kx[M[0]] - kx_dft[np.argmin(np.abs(kx_dft))]) / (MP_dft[0] - np.argmin(np.abs(kx_dft)))) * (MP_dft[0] - 1)
kx_dft_step = (kx[M[0]] - kx_dft[np.argmin(np.abs(kx_dft))]) / (MP_dft[0] - np.argmin(np.abs(kx_dft)))
ky_dft_0 = ky[M[1]] - ((ky[M[1]] - ky_dft[np.argmin(np.abs(ky_dft))]) / (MP_dft[1] - np.argmin(np.abs(ky_dft)))) * (MP_dft[1] - 1)
ky_dft_step = (ky[M[1]] - ky_dft[np.argmin(np.abs(ky_dft))]) / (MP_dft[1] - np.argmin(np.abs(ky_dft)))

kx_dft_a = np.array(np.arange(Decimal(kx_dft_0), Decimal(kx_dft_0 + kx_dft.shape[0] * kx_dft_step), Decimal(kx_dft_step)), dtype="float64")
ky_dft_a = np.array(np.arange(Decimal(ky_dft_0), Decimal(ky_dft_0 + ky_dft.shape[0] * ky_dft_step), Decimal(ky_dft_step)), dtype="float64")

E_dft_a = np.zeros((E_dft.shape[0], kx.shape[0], ky.shape[0]))
for ind in range(E_dft.shape[0]):
    E_dft_a[ind] = theory_data_expand(ind, kx_dft_a, ky_dft_a, E_dft, kx, ky, kx.shape[0])

n_bands = len(HYPERPARAMS)
recon = np.zeros((n_bands, kx.shape[0], ky.shape[0]))

for ind_band in range(n_bands):
    hp = HYPERPARAMS[ind_band]
    print(f"Band {ind_band}: eta={hp['eta']}")
    mrf.eta = hp["eta"]
    Einterp = theory_data_expand(ind_band * 2, kx_dft_a, ky_dft_a, E_dft_a, kx, ky, kx.shape[0])
    E0 = np.reshape(Einterp + hp["offset"], (kx.shape[0], ky.shape[0]))
    EE, EE0 = np.meshgrid(E, E0)
    mrf.indEb = np.argmin(np.abs(EE - EE0), 1).reshape(E0.shape)
    recon[ind_band] = mrf.getEb()
    sym_band(ind_band, recon, kx, ky, mrf.lengthKx, mrf.lengthKy)

np.save(os.path.join(TEST_DIR, "recon_bands.npy"), recon)
print(f"Recon saved: {recon.shape}")

---
## Step 8: 可视化

In [ ]:
print("=" * 60)
print("Step 8: Visualizations")
print("=" * 60)
colors = ["r", "y", "b", "g", "w"]

# 重建能带总览
fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 4))
if n_bands == 1: axes = [axes]
for i in range(n_bands):
    im = axes[i].imshow(recon[i], extent=(ky[0], ky[-1], kx[0], kx[-1]),
                        origin="lower", cmap="viridis", aspect="auto")
    axes[i].set(title=f"Band {i}", xlabel=r"$k_y$ ($\AA^{-1}$)", ylabel=r"$k_x$ ($\AA^{-1}$)")
    plt.colorbar(im, ax=axes[i], label="E (eV)")
plt.tight_layout()
fig.savefig(os.path.join(TEST_DIR, "recon_bands.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Γ-M-K-Γ 路径能带图
nGM = int(math.sqrt((M[0] - G[0]) ** 2 + (M[1] - G[1]) ** 2))
nMK = int(math.sqrt((M[0] - K[0]) ** 2 + (M[1] - K[1]) ** 2))
nKG = int(math.sqrt((K[0] - G[0]) ** 2 + (K[1] - G[1]) ** 2))

path_points = np.asarray([G, M, K, G])
row_inds, col_inds, path_inds = points2path(path_points[:, 0], path_points[:, 1],
                                              npoints=[nGM, nMK, nKG])
pathD = bpm(np.transpose(I, (2, 0, 1)), pathr=row_inds, pathc=col_inds, eaxis=0)
prec = bpm(recon, pathr=row_inds, pathc=col_inds, eaxis=0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(pathD, cmap="plasma",
          extent=[0, len(row_inds), E[0], E[min(109, len(E) - 1)]], aspect="auto", origin="upper")
for ib in range(prec.shape[0]):
    ax.plot(savgol_filter(prec[ib], min(30, len(prec[ib]) - 2), 2),
            zorder=1, lw=2.3, color=colors[ib], label=f"Band {ib}")
ax.set(xlim=(0, len(row_inds)), ylim=(E[0], E[min(109, len(E) - 1)]))
ax.set_xticks(path_inds)
ax.set_xticklabels([r"$\overline{\Gamma}$", r"$\overline{\mathrm{M}}$",
                     r"$\overline{\mathrm{K}}$", r"$\overline{\Gamma}$"], fontsize=15)
ax.legend()
plt.colorbar(ax.images[0], label="Intensity")
fig.savefig(os.path.join(TEST_DIR, "band_path_GMKG.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# DFT vs 重建对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
dft_path = np.vstack((ebands[0, :, 2:], ebands[:, -1, 2:], ebands[-1, ::-1, 2:]))
for band in range(105, 130):
    ax1.plot(dft_path[:, band], color="blue", alpha=0.3, lw=0.5)
ax1.set(ylabel="E-EF (eV)", xlabel="k-path", title="DFT Bands (GMKG)", ylim=(-2, 2))
ax1.axhline(0, color="gray", ls="--", lw=0.5)
for ib in range(prec.shape[0]):
    ax2.plot(prec[ib], color=colors[ib], lw=2, label=f"Band {ib}")
ax2.set(ylabel="E (eV)", xlabel="k-path", title="Reconstructed Bands (GMKG)")
ax2.legend()
plt.tight_layout()
fig.savefig(os.path.join(TEST_DIR, "dft_vs_recon.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 总结

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"Data source: {EXP_H5} → {EXP_DSET}")
print(f"Fermi: {fermi:.4f} eV, VBM={vbm}, CBM={cbm}")
print(f"DFT: 20x20 → BZ {bz_coords.shape[0]} pts → mapping {mapping.shape}")
print(f"Exp: {bands.shape} → preprocessed {E_Mon.shape}")
print(f"MRF: {n_bands} bands, kx={kx.shape}, ky={ky.shape}")
print(f"G={G}, M={M}, K={K}")
print(f"Calibration: kx_shift={calib['kx_shift']:.4f}, ky_shift={calib['ky_shift']:.4f}")
print(f"Output: {TEST_DIR}/")
for f in sorted(os.listdir(TEST_DIR)):
    sz = os.path.getsize(os.path.join(TEST_DIR, f))
    print(f"  {f:40s} {sz:>12,d} bytes")